# Ensemble Ablation — 14 Model

VGG16, ResNet50, NASNetMobile, Xception, ViT-Base, ViT-Large, ViT-Huge, Swin-Tiny, DeiT ve CvT-13 

A single 80%/10%/10% train/validation/test split is used, with the same saved indices shared across all models. The eight CNN backbones are trained with their parameters unfrozen. The six Transformer backbones serve as fixed feature extractors, while their Keras classifiers are trained. Transformers are loaded sequentially through PyTorch; CNNs use TensorFlow/Keras. In this version, the DeiT backbone has also been migrated to the PyTorch backend; bitwise-identical results to the previous TensorFlow implementation are not guaranteed.

AB_MODE="compact" includes individual models, leave-one-model-out ensembles, the full ensemble, CNN-only and Transformer-only ensembles, and the original CVE/CTVE groups: 33 groups × 2 voting strategies = 66 rows. AB_MODE="all" evaluates all non-empty subsets, producing 32,766 rows and correspondingly more image-level rows in the test reports.

The original training settings are a batch size of 32, 30 epochs, and the Adam optimizer with a learning rate of 1e-4. DeiT uses a feature extraction batch size of 16 and a prediction batch size of 8. The newly added ViT-Huge and ViT-Large use feature extraction batch sizes of 1 and 2, respectively. Larger models may run more slowly. Pretrained weights are downloaded on the first run. The selected checkpoints use different pretraining datasets; their sources are explicitly listed in TRANSFORMER_SPECS.

For 224×224 inputs, the default NASNet variant is NASNetMobile.


In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

## 1. Working GPU Environment

Select DEIT GPU — TensorFlow + Torch as the kernel. The verified environment uses Python 3.11, TensorFlow 2.16.1, tf-keras 2.16.0, and Torch 2.7.1+cu118. This notebook does not modify package versions. Transformers 4.44.2, NumPy, scikit-learn, Pillow, and pandas are required.

Specify the data directory in the configuration settings. Run each dataset separately and restart the kernel when switching datasets. Execute the cells sequentially from beginning to end.

In [2]:
import sys
from importlib import metadata
print("Python:", sys.executable)
for package in ("tensorflow", "tf-keras", "torch", "transformers", "numpy"):
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "KURULU DEĞİL")
# Çalışan ortamı korumak için burada pip install çalıştırılmaz.

Python: /home/tubitak1001/projects/deit_env/bin/python
tensorflow 2.16.1
tf-keras 2.16.0
torch 2.7.1+cu118
transformers 4.46.3
numpy 1.26.4


## 2. Settings

In [2]:
from pathlib import Path
from types import SimpleNamespace

DATA_ROOT = Path("/home/tubitak1001/Desktop/arzu")
DATASET_NAME = "data_arzu"  # data_arzu, data_yushu, data_ludian, data_other
BASE_DIR = DATA_ROOT / DATASET_NAME

CLASSES = ["damaged", "undamaged"]
POSITIVE_CLASS = "damaged"
OUTPUT_DIR = DATA_ROOT / f"{DATASET_NAME}_ablasyon_14models_v2"

SPLIT_DIR = DATA_ROOT / f"{DATASET_NAME}_14models_split_80_10_10"
GROUPS_CSV = None  # Example: Path("groups.csv"); sütunlar: path,group
SEED = 42
EPOCHS = 30
BATCH_SIZE = 32
PREDICT_BATCH_SIZE = 8
TRAIN_FRACTION = 0.80
VAL_FRACTION = 0.10
TEST_FRACTION = 0.10
DEIT_FEATURE_BATCH_SIZE = 16
MODEL_NAMES = ['VGG19', 'DenseNet201', 'DenseNet169', 'ConvNeXtBase', 'VGG16', 'ResNet50', 'NASNetMobile', 'Xception', 'DeiT', 'ViTBase', 'ViTLarge', 'ViTHuge', 'SwinT', 'CvT']
CNN_NAMES = ['VGG19', 'DenseNet201', 'DenseNet169', 'ConvNeXtBase', 'VGG16', 'ResNet50', 'NASNetMobile', 'Xception']
AB_MODE = "compact" 
TRANSFORMER_DEVICE = "cuda"
TRANSFORMER_BATCH_SIZES = {"ViTHuge": 1, "ViTLarge": 2}

TRANSFORMER_SPECS = {
    "DeiT": "facebook/deit-base-distilled-patch16-224",
    "ViTBase": "google/vit-base-patch16-224-in21k",
    "ViTLarge": "google/vit-large-patch16-224-in21k",
    "ViTHuge": "google/vit-huge-patch14-224-in21k",
    "SwinT": "microsoft/swin-tiny-patch4-window7-224",
    "CvT": "microsoft/cvt-13",
}

TIE_RULE = "lowest"  
EVALUATE_ONLY = False  

args = SimpleNamespace(
    base_dir=BASE_DIR, classes=CLASSES,
    positive_class=POSITIVE_CLASS, groups_csv=GROUPS_CSV,
    output=OUTPUT_DIR, split_dir=SPLIT_DIR, seed=SEED, epochs=EPOCHS, batch_size=BATCH_SIZE,
    predict_batch_size=PREDICT_BATCH_SIZE, train_fraction=TRAIN_FRACTION,
    val_fraction=VAL_FRACTION, test_fraction=TEST_FRACTION,
    deit_feature_batch_size=DEIT_FEATURE_BATCH_SIZE,
    tie_rule=TIE_RULE, evaluate_only=EVALUATE_ONLY,
    ab_mode=AB_MODE, transformer_device=TRANSFORMER_DEVICE,
    transformer_batch_sizes=TRANSFORMER_BATCH_SIZES,
)
output = args.output.expanduser().resolve()
print("Çıktı klasörü:", output)


Çıktı klasörü: /home/tubitak1001/Desktop/arzu/data_arzu_ablasyon_14models_v2


The shared 80/10/10 split is saved in SPLIT_DIR. On subsequent runs, if the data list, classes, group information, and image contents remain unchanged, the saved indices are loaded; the data are not split again. For different training runs, change only OUTPUT_DIR and keep SPLIT_DIR unchanged.

This directory contains image copies under train/<class>/, val/<class>/, and test/<class>/, along with split_indices.npz, split_manifest.csv, sample_index.csv, class_counts.csv, and split_metadata.json. Images are copied in their original file format and resolution and resized to 224×224 during training. If the source images are already 224×224, the copies retain these dimensions.

The indices refer to the shared data array after unreadable and duplicate images have been removed. GROUPS_CSV can be used to keep images of the same building or scene within the same split. If the data change, the previous split is not reused, and you are prompted to select a new SPLIT_DIR.


## 3. Metrics and voting functions

In [2]:
import argparse
import csv
import gc
import itertools
import hashlib
import json
import os
import shutil
from pathlib import Path
import numpy as np
from PIL import Image
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, matthews_corrcoef, precision_score, recall_score, roc_auc_score
METRIC_NAMES = ['accuracy', 'balanced_accuracy', 'sensitivity', 'specificity', 'precision', 'f1_positive', 'f1_macro', 'mcc', 'auc', 'tn', 'fp', 'fn', 'tp', 'delta_accuracy_vs_full', 'delta_sensitivity_vs_full', 'delta_specificity_vs_full', 'delta_f1_positive_vs_full', 'delta_mcc_vs_full']

def validate(probabilities, labels, names, classes):
    if probabilities.ndim != 3 or probabilities.shape != (len(names), len(labels), 2):
        raise ValueError('Expected probabilities [models, samples, 2]')
    if len(classes) != 2 or len(set(names)) != len(names) or len(names) < 2:
        raise ValueError('Need two classes and at least two distinct model names')
    if not np.array_equal(np.unique(labels), [0, 1]):
        raise ValueError('Test data must contain both classes, encoded 0 and 1')
    if not np.isfinite(probabilities).all() or np.any(probabilities < 0) or np.any(probabilities > 1):
        raise ValueError('Invalid probabilities')
    if not np.allclose(probabilities.sum(axis=2), 1, atol=0.0001):
        raise ValueError('Probability rows must sum to one')

def vote(probabilities, strategy, tie_rule):
    if strategy == 'soft':
        scores = probabilities.mean(axis=0)
        return (scores.argmax(axis=1), scores, 0)
    votes = probabilities.argmax(axis=2)
    counts = np.stack([(votes == c).sum(axis=0) for c in range(2)], axis=1)
    scores = counts / len(probabilities)
    predicted = counts.argmax(axis=1)
    ties = counts[:, 0] == counts[:, 1]
    if tie_rule == 'soft':
        predicted[ties] = probabilities.mean(axis=0)[ties].argmax(axis=1)
    return (predicted, scores, int(ties.sum()))

def metrics(labels, predicted, scores, positive):
    actual = (labels == positive).astype(int)
    estimated = (predicted == positive).astype(int)
    tn, fp, fn, tp = confusion_matrix(actual, estimated, labels=[0, 1]).ravel()
    return {'accuracy': accuracy_score(actual, estimated), 'balanced_accuracy': balanced_accuracy_score(actual, estimated), 'sensitivity': recall_score(actual, estimated, zero_division=0), 'specificity': tn / (tn + fp), 'precision': precision_score(actual, estimated, zero_division=0), 'f1_positive': f1_score(actual, estimated, zero_division=0), 'f1_macro': f1_score(actual, estimated, average='macro', zero_division=0), 'mcc': matthews_corrcoef(actual, estimated), 'auc': roc_auc_score(actual, scores[:, positive]), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}

def evaluate(probabilities, labels, names, classes, output, positive_name, tie_rule, provenance):
    validate(probabilities, labels, names, classes)
    if positive_name not in classes:
        raise ValueError(f'Positive class {positive_name!r} absent; classes: {list(classes)}')
    positive = list(classes).index(positive_name)
    rows = []
    for indices in experiment_subsets(names):
        size = len(indices)
        selected = [names[i] for i in indices]
        removed = [n for n in names if n not in selected]
        if size == len(names):
            kind = 'full_ensemble'
        elif size == 1:
            kind = 'single_model'
        elif set(selected) == set(name for name in names if name in CNN_NAMES):
            kind = 'CNN_only'
        elif set(selected) == set(name for name in names if name not in CNN_NAMES):
            kind = 'Transformer_only'
        elif set(selected) == {'VGG19', 'DenseNet201', 'DenseNet169', 'ConvNeXtBase', 'DeiT'}:
            kind = 'original_CTVE'
        elif set(selected) == {'VGG19', 'DenseNet201', 'DenseNet169', 'ConvNeXtBase'}:
            kind = 'original_CVE'
        elif len(removed) == 1:
            kind = 'leave_one_out'
        else:
            kind = 'subset'
        for strategy in ('soft', 'hard'):
            predicted, scores, ties = vote(probabilities[list(indices)], strategy, tie_rule)
            row = {'experiment': kind, 'models': '+'.join(selected), 'removed': '+'.join(removed), 'n_models': size, 'voting': strategy, 'positive_class': positive_name, 'tie_rule': tie_rule if strategy == 'hard' else 'lowest_argmax', 'tie_count': ties, 'n_test': len(labels)}
            row.update(metrics(labels, predicted, scores, positive))
            rows.append(row)
    for row in rows:
        baseline = next((r for r in rows if r['experiment'] == 'full_ensemble' and r['voting'] == row['voting']))
        for key in ('accuracy', 'sensitivity', 'specificity', 'f1_positive', 'mcc'):
            row['delta_' + key + '_vs_full'] = row[key] - baseline[key]
    with (output / 'ablation_results.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    metadata = {'provenance': provenance, 'class_mapping': dict(enumerate(map(str, classes))), 'positive_class': positive_name, 'hard_tie_rule': tie_rule, 'delta_definition': 'subset metric minus full ensemble with same voting', 'auc_definition': 'soft: mean class probability; hard: positive vote fraction', 'warning': 'Descriptive test ablation; do not select subsets using test results.', 'experiments': len(rows)}
    (output / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    print('\nComponent ablation (key comparisons):')
    for r in rows:
        if r['experiment'] != 'subset':
            print(f"{r['models']:55s} {r['voting']:4s} Acc={r['accuracy']:.4f} Sens={r['sensitivity']:.4f} Spec={r['specificity']:.4f} F1={r['f1_positive']:.4f} MCC={r['mcc']:.4f} deltaAcc={r['delta_accuracy_vs_full']:+.4f}")
    print(f"Saved {len(rows)} rows to {output / 'ablation_results.csv'}")
    return rows

def experiment_subsets(names):
    n = len(names)
    if AB_MODE == 'all':
        return [indices for size in range(1, n + 1) for indices in itertools.combinations(range(n), size)]
    if AB_MODE != 'compact':
        raise ValueError('AB_MODE compact veya all olmalı.')
    selections = [(i,) for i in range(n)]
    selections += [tuple(range(n))]
    selections += [tuple((i for i in range(n) if i != dropped)) for dropped in range(n)]
    selections += [tuple((i for i, name in enumerate(names) if name in CNN_NAMES)), tuple((i for i, name in enumerate(names) if name not in CNN_NAMES))]
    for original in [('VGG19', 'DenseNet201', 'DenseNet169', 'ConvNeXtBase', 'DeiT'), ('VGG19', 'DenseNet201', 'DenseNet169', 'ConvNeXtBase')]:
        if set(original) <= set(names):
            selections.append(tuple((i for i, name in enumerate(names) if name in original)))
    return list(dict.fromkeys((indices for indices in selections if indices)))

## 4. Model Learning Functions for Eight CNN ve six Transformer

The Transformer backbones are frozen, and the same BatchNorm → Dense(512) → Dropout → Dense(256) → Dropout → Softmax classification head is trained. The CNN training policy remains unchanged.

In [3]:
def create_model_nn(base, preprocess_fn):
    inp = Input(shape=(*img_size, 3))
    x = preprocess_fn(inp)
    x = base(x, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(1024, activation='relu')(x)
    x = Dropout(0.4)(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.4)(x)
    out = Dense(len(class_names), activation='softmax')(x)
    return Model(inp, out)

def train_cnn_model(base, preprocess_fn, name, epochs=30, batch_size=32):
    model = create_model_nn(base, preprocess_fn)
    model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    ckpt = ModelCheckpoint(f"{name}.h5", monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    rl = ReduceLROnPlateau(monitor='val_loss', patience=5, factor=0.3, verbose=1)
    model.fit(X_train, y_train_oh, validation_data=(X_val, y_val_oh),
              epochs=epochs, batch_size=batch_size, callbacks=[ckpt, es, rl], verbose=1)
    return model

def train_deit_classifier(train_feats, val_feats, name, epochs=30, batch_size=32):
    input_shape = train_feats.shape[1]
    inp = Input(shape=(input_shape,))
    x = BatchNormalization()(inp)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x)
    out = Dense(len(class_names), activation='softmax')(x)
    clf = Model(inp, out)
    clf.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    ckpt = ModelCheckpoint(f"{name}_deit_clf.h5", monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    rl = ReduceLROnPlateau(monitor='val_loss', patience=5, factor=0.3, verbose=1)
    clf.fit(train_feats, y_train_oh, validation_data=(val_feats, y_val_oh),
            epochs=epochs, batch_size=batch_size, callbacks=[ckpt, es, rl], verbose=1)
    return clf

def predict_in_batches_model(model, X, batch_size=8):
    N = X.shape[0]
    probs = np.zeros((N, len(class_names)), dtype=np.float32)
    for i in range(0, N, batch_size):
        end = min(i + batch_size, N)
        probs[i:end] = model.predict(X[i:end], batch_size=end-i, verbose=0)
    return probs

def init_training():
    # Lazy imports allow --evaluate-only on a CPU environment without TF.
    imports = '''import tensorflow as tf
from tensorflow.keras.applications import VGG19, DenseNet201, DenseNet169, ConvNeXtBase, VGG16, ResNet50, NASNetMobile, Xception
from tensorflow.keras.applications.vgg19 import preprocess_input as vgg19_preprocess
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.convnext import preprocess_input as convnext_preprocess
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet50_preprocess
from tensorflow.keras.applications.nasnet import preprocess_input as nasnet_preprocess
from tensorflow.keras.applications.xception import preprocess_input as xception_preprocess
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
'''
    exec(imports, globals())
    for gpu in tf.config.list_physical_devices('GPU'):
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as exc:
            print(f'GPU memory configuration: {exc}')



def extract_transformer_features(name, images, args):
    # HF PyTorch backend covers ViT-Huge and CvT without TF model conversion.
    import torch
    from transformers import AutoImageProcessor, AutoModel
    checkpoint = TRANSFORMER_SPECS[name]
    device = torch.device(args.transformer_device)
    batch_size = args.transformer_batch_sizes.get(name, args.deit_feature_batch_size)
    if batch_size < 1:
        raise ValueError("Transformer batch size pozitif olmalı.")
    print(f"{name}: {checkpoint}, cihaz={device}, batch={batch_size}", flush=True)
    processor = AutoImageProcessor.from_pretrained(checkpoint)
    backbone = AutoModel.from_pretrained(checkpoint).to(device)
    backbone.eval()
    backbone.requires_grad_(False)
    output_features = None
    try:
        with torch.inference_mode():
            for start in range(0, len(images), batch_size):
                end = min(start + batch_size, len(images))
                batch = [Image.fromarray(np.clip(image, 0, 255).astype(np.uint8))
                         for image in images[start:end]]
                inputs = processor(images=batch, return_tensors="pt")
                inputs = {key: value.to(device) for key, value in inputs.items()}
                hidden = backbone(**inputs).last_hidden_state
                if hidden.ndim == 3:
                    pooled = hidden.mean(dim=1)  # Same mean-token pooling as original DeiT.
                elif hidden.ndim == 4:
                    pooled = hidden.mean(dim=(2, 3))  # CvT: channels-first feature map.
                else:
                    raise ValueError(f"Beklenmeyen özellik boyutu: {hidden.shape}")
                features = pooled.detach().cpu().numpy().astype(np.float32)
                if output_features is None:
                    output_features = np.empty((len(images), features.shape[1]), dtype=np.float32)
                output_features[start:end] = features
                del inputs, hidden, pooled, features, batch
                if end == len(images) or start % (batch_size * 20) == 0:
                    print(f"{name}: {end}/{len(images)} özellik çıkarıldı", flush=True)
        return output_features
    finally:
        del backbone, processor
        if device.type == "cuda":
            torch.cuda.empty_cache()
        elif device.type == "mps":
            torch.mps.empty_cache()
        gc.collect()


"""def train_single_split(images, labels, split, classes, args, directory):
    global X_train, X_val, X_test, y_train_oh, y_val_oh, class_names, img_size
    train, val, test = split
    class_names, img_size = classes, (224, 224)
    tf.keras.utils.set_random_seed(args.seed)
    X_train, X_val, X_test = images[train], images[val], images[test]
    y_train_oh = tf.keras.utils.to_categorical(labels[train], 2)
    y_val_oh = tf.keras.utils.to_categorical(labels[val], 2)
    factories = {
        "VGG19": (VGG19, vgg19_preprocess),
        "DenseNet201": (DenseNet201, densenet_preprocess),
        "DenseNet169": (DenseNet169, densenet_preprocess),
        "ConvNeXtBase": (ConvNeXtBase, convnext_preprocess),
        "VGG16": (VGG16, vgg16_preprocess),
        "ResNet50": (ResNet50, resnet50_preprocess),
        "NASNetMobile": (NASNetMobile, nasnet_preprocess),
        "Xception": (Xception, xception_preprocess),
    }
    predictions = []
    previous = Path.cwd()
    directory = Path(directory).resolve()
    try:
        os.chdir(directory)
        for position, name in enumerate(MODEL_NAMES, 1):
            print(f"\nModel {position}/{len(MODEL_NAMES)}: {name}", flush=True)
            tf.keras.backend.clear_session()
            if name in factories:
                factory, preprocess = factories[name]
                base = factory(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
                model = train_cnn_model(base, preprocess, name, epochs=args.epochs, batch_size=args.batch_size)
                probabilities = predict_in_batches_model(model, X_test, batch_size=args.predict_batch_size)
                del model, base
            else:
                features = extract_transformer_features(name, images, args)
                classifier = train_deit_classifier(features[train], features[val], name,
                                                   epochs=args.epochs, batch_size=args.batch_size)
                probabilities = classifier.predict(features[test], batch_size=args.predict_batch_size, verbose=0)
                del classifier, features
            # Save immediately: already finished branch predictions survive interruption.
            np.savez_compressed(directory / (name + "_test_predictions.npz"),
                                probabilities=probabilities, labels=labels[test],
                                test_indices=test, model_name=name)
            predictions.append(probabilities)
            del probabilities
            tf.keras.backend.clear_session()
            gc.collect()
    finally:
        os.chdir(previous)
        X_train = X_val = X_test = y_train_oh = y_val_oh = None
    return np.stack(predictions)
"""
def train_single_split(images, labels, split, classes, args, directory):
    global X_train, X_val, X_test
    global y_train_oh, y_val_oh, class_names, img_size

    train, val, test = split
    class_names, img_size = classes, (224, 224)

    tf.keras.utils.set_random_seed(args.seed)

    X_train = images[train]
    X_val = images[val]
    X_test = images[test]

    y_train_oh = tf.keras.utils.to_categorical(labels[train], 2)
    y_val_oh = tf.keras.utils.to_categorical(labels[val], 2)

    factories = {
        "VGG19": (VGG19, vgg19_preprocess),
        "DenseNet201": (DenseNet201, densenet_preprocess),
        "DenseNet169": (DenseNet169, densenet_preprocess),
        "ConvNeXtBase": (ConvNeXtBase, convnext_preprocess),
        "VGG16": (VGG16, vgg16_preprocess),
        "ResNet50": (ResNet50, resnet50_preprocess),
        "NASNetMobile": (NASNetMobile, nasnet_preprocess),
        "Xception": (Xception, xception_preprocess),
    }

    predictions = []
    timings = []

    previous = Path.cwd()
    directory = Path(directory).resolve()

    try:
        os.chdir(directory)

        for position, name in enumerate(MODEL_NAMES, 1):
            print(f"\nModel {position}/{len(MODEL_NAMES)}: {name}")
            tf.keras.backend.clear_session()

            feature_extract_all_sec = 0.0

            if name in factories:
                factory, preprocess = factories[name]

                # Omurga yükleme süresi eğitim ölçümüne dahil değil.
                base = factory(
                    weights="imagenet",
                    include_top=False,
                    input_shape=(224, 224, 3),
                )

                start = time.perf_counter()
                model = train_cnn_model(
                    base, preprocess, name,
                    epochs=args.epochs,
                    batch_size=args.batch_size,
                )
                train_time_sec = time.perf_counter() - start

                start = time.perf_counter()
                probabilities = predict_in_batches_model(
                    model, X_test,
                    batch_size=args.predict_batch_size,
                )
                test_time_sec = time.perf_counter() - start

                test_time_scope = "CNN image-to-probability"
                del model, base

            else:
                start = time.perf_counter()
                features = extract_transformer_features(
                    name, images, args
                )
                feature_extract_all_sec = (
                    time.perf_counter() - start
                )

                start = time.perf_counter()
                classifier = train_deit_classifier(
                    features[train],
                    features[val],
                    name,
                    epochs=args.epochs,
                    batch_size=args.batch_size,
                )
                train_time_sec = time.perf_counter() - start

                start = time.perf_counter()
                probabilities = classifier.predict(
                    features[test],
                    batch_size=args.predict_batch_size,
                    verbose=0,
                )
                test_time_sec = time.perf_counter() - start

                test_time_scope = "Transformer classifier on precomputed features"
                del classifier, features

            timings.append({
                "model": name,
                "train_time_sec": train_time_sec,
                "test_time_sec": test_time_sec,
                "test_time_per_image_sec": test_time_sec / len(test),
                "feature_extract_all_sec": feature_extract_all_sec,
                "test_time_scope": test_time_scope,
                "n_test": len(test),
            })

            # Her tamamlanan modelden sonra süreleri kaydet.
            import pandas as pd
            pd.DataFrame(timings).to_csv(
                directory / "model_timings.csv",
                index=False,
            )

            np.savez_compressed(
                directory / f"{name}_test_predictions.npz",
                probabilities=probabilities,
                labels=labels[test],
                test_indices=test,
                model_name=name,
            )

            predictions.append(probabilities)

            print(
                f"{name}: train={train_time_sec:.2f} sn, "
                f"test={test_time_sec:.2f} sn, "
                f"özellik çıkarma={feature_extract_all_sec:.2f} sn"
            )

            del probabilities
            tf.keras.backend.clear_session()
            gc.collect()

    finally:
        os.chdir(previous)
        X_train = X_val = X_test = None
        y_train_oh = y_val_oh = None

    return np.stack(predictions)

## 5. Data split functions

In [4]:
def load_dataset(args):
    root = args.base_dir.expanduser().resolve()
    classes = sorted(args.classes)
    folders = [root]
    group_map = None
    if args.groups_csv:
        with args.groups_csv.open(newline='', encoding='utf-8') as f:
            records = list(csv.DictReader(f))
        if not records or not {'path', 'group'} <= records[0].keys():
            raise ValueError('groups CSV must have path,group columns')
        group_map = {}
        for row in records:
            key = row['path'].replace('\\', '/')
            if not row['group'] or key in group_map:
                raise ValueError(f'Empty or duplicate group mapping: {key}')
            group_map[key] = row['group']
    images, labels, paths, groups, hashes = ([], [], [], [], [])
    seen, rejected, duplicates = ({}, [], [])
    for folder in folders:
        for label, cls in enumerate(classes):
            directory = folder / cls
            if not directory.is_dir():
                raise ValueError(f'Missing class directory: {directory}')
            for path in sorted(directory.rglob('*')):
                if not path.is_file() or path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}:
                    continue
                rel = path.relative_to(root).as_posix()
                try:
                    with Image.open(path) as im:
                        im = im.convert('RGB')
                        digest = hashlib.sha256(str(im.size).encode() + im.tobytes()).hexdigest()
                        arr = np.asarray(im.resize((224, 224), Image.Resampling.NEAREST), dtype=np.float32)
                except (OSError, ValueError) as exc:
                    rejected.append({'path': rel, 'reason': str(exc)})
                    continue
                group = group_map.get(rel) if group_map is not None else None
                if group_map is not None and group is None:
                    raise ValueError(f'Missing group for {rel}')
                if digest in seen:
                    previous = seen[digest]
                    if labels[previous] != label:
                        raise ValueError(f'Identical images have conflicting labels: {rel}, {paths[previous]}')
                    if group_map is not None and groups[previous] != group:
                        raise ValueError(f'Identical images must share a group: {rel}, {paths[previous]}')
                    duplicates.append({'removed': rel, 'kept': paths[previous]})
                    continue
                seen[digest] = len(images)
                images.append(arr)
                labels.append(label)
                paths.append(rel)
                groups.append(group)
                hashes.append(digest)
    if not images:
        raise ValueError('No readable images found')
    return (np.stack(images), np.asarray(labels), paths, np.asarray(groups) if group_map is not None else None, classes, {'rejected': rejected, 'duplicates_removed': duplicates, 'image_sha256': hashes})

def make_single_split(labels, groups, seed):
    indices = np.arange(len(labels))
    if groups is None:
        outer = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=seed)
        remaining, test = next(outer.split(indices, labels))
        val_count = int(np.ceil(0.1 * len(labels)))
        inner = StratifiedShuffleSplit(n_splits=1, test_size=val_count, random_state=seed + 1)
        train_local, val_local = next(inner.split(remaining, labels[remaining]))
    else:
        if len(np.unique(groups)) < 10:
            raise ValueError('Gruplu 80/10/10 bölme için en az 10 bağımsız grup gerekli.')
        outer = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=seed)
        remaining, test = next(outer.split(indices, labels, groups))
        if len(np.unique(groups[remaining])) < 9:
            raise ValueError('Validation bölmesi için kalan havuzda en az 9 grup gerekli.')
        inner = StratifiedGroupKFold(n_splits=9, shuffle=True, random_state=seed + 1)
        train_local, val_local = next(inner.split(remaining, labels[remaining], groups[remaining]))
    train, val = (remaining[train_local], remaining[val_local])
    for part in (train, val, test):
        if not np.array_equal(np.unique(labels[part]), [0, 1]):
            raise ValueError('Her bölüm iki sınıfı da içermeli; veri/grup sayısını kontrol edin.')
        part.setflags(write=False)
    for a, b in ((train, val), (train, test), (val, test)):
        assert not set(a) & set(b)
        if groups is not None:
            assert not set(groups[a]) & set(groups[b])
    assert sorted(np.concatenate([train, val, test]).tolist()) == indices.tolist()
    return (train, val, test)

def write_csv(path, rows):
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def evaluate_single_split(args, output):
    with np.load(output / 'predictions.npz', allow_pickle=False) as data:
        probabilities, labels = (data['probabilities'], data['labels'])
        names, classes = (data['model_names'].tolist(), data['class_names'].tolist())
        provenance = json.loads(data['provenance'].item())
        if 'test_indices' not in data.files or 'test_paths' not in data.files:
            raise ValueError('predictions.npz dosyasında test_indices/test_paths yok; görüntü eşlemesi yapılamaz.')
        test_indices = data['test_indices'].copy()
        test_paths = data['test_paths'].tolist()
    if provenance.get('protocol') != 'single_split_80_10_10':
        raise ValueError('Bu dosya tek split 80/10/10 çalışmasına ait değil.')
    rows = evaluate(probabilities, labels, names, classes, output, args.positive_class, args.tie_rule, provenance)
    export_test_predictions(probabilities, labels, names, classes, test_indices, test_paths, output, args.positive_class, args.tie_rule)
    return rows

def validate_saved_split(split, labels, groups):
    merged = []
    for part in split:
        if part.ndim != 1 or not np.issubdtype(part.dtype, np.integer):
            raise ValueError('İndeks dizisinin biçimi geçersiz.')
        if np.any(part < 0) or np.any(part >= len(labels)):
            raise ValueError('Kayıtlı indeks veri aralığı dışında.')
        if len(np.unique(part)) != len(part):
            raise ValueError('Kayıtlı indekste tekrar var.')
        if not np.array_equal(np.unique(labels[part]), [0, 1]):
            raise ValueError('Her bölüm her iki sınıfı içermeli.')
        part.setflags(write=False)
        merged.extend(part.tolist())
    if sorted(merged) != list(range(len(labels))):
        raise ValueError('Kayıtlı bölme ayrık değil veya tüm örnekleri kapsamıyor.')
    if groups is not None:
        for a, b in ((split[0], split[1]), (split[0], split[2]), (split[1], split[2])):
            if set(groups[a]) & set(groups[b]):
                raise ValueError('Kayıtlı bölmede gruplar çakışıyor.')

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def prepare_reusable_split(args, labels, paths, groups, classes, audit):
    source = args.base_dir.expanduser().resolve()
    directory = args.split_dir.expanduser().resolve()
    if source == directory or source in directory.parents or directory in source.parents:
        raise ValueError('SPLIT_DIR ile BASE_DIR birbirinden ayrı klasörler olmalı.')
    identity = {'protocol': 'single_split_80_10_10', 'classes': classes, 'paths': paths, 'labels': labels.tolist(), 'image_sha256': audit['image_sha256'], 'groups': groups.tolist() if groups is not None else None}
    fingerprint = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    metadata_path = directory / 'split_metadata.json'
    index_path = directory / 'split_indices.npz'
    if metadata_path.exists() and index_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        if metadata.get('dataset_fingerprint') != fingerprint:
            raise ValueError('Veri/sınıflar/gruplar değişti. Yeni bir SPLIT_DIR seçin.')
        with np.load(index_path, allow_pickle=False) as saved:
            split = tuple((saved[key].copy() for key in ('train', 'validation', 'test')))
        if metadata.get('indices_sha256') != file_sha256(index_path):
            raise ValueError('Kayıtlı split_indices.npz değişmiş; bölme dosyasını kontrol edin.')
        validate_saved_split(split, labels, groups)
        print('Kayıtlı train/val/test indeksleri yeniden kullanılıyor:', index_path, flush=True)
    else:
        if directory.exists() and any(directory.iterdir()):
            raise ValueError('SPLIT_DIR dolu fakat geçerli bölme dosyaları yok. Yeni bir klasör seçin.')
        split = make_single_split(labels, groups, args.seed)
        directory.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(index_path, train=split[0], validation=split[1], test=split[2])
        metadata = {'dataset_fingerprint': fingerprint, 'split_seed': args.seed, 'fractions': [0.8, 0.1, 0.1], 'identity': identity, 'indices_sha256': file_sha256(index_path)}
        metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
        print('Yeni ortak indeksler oluşturuldu ve kaydedildi:', index_path, flush=True)
    manifest = []
    counts = [{'partition': 'all', 'total': len(labels), 'percent': 100.0, **{name: int(np.sum(labels == i)) for i, name in enumerate(classes)}}]
    for role, indices in zip(('train', 'val', 'test'), split):
        for name in classes:
            (directory / role / name).mkdir(parents=True, exist_ok=True)
        for number, i in enumerate(indices, 1):
            relative = Path(paths[i])
            original = source / relative
            copied = directory / role / relative
            copied.parent.mkdir(parents=True, exist_ok=True)
            if copied.exists():
                if file_sha256(original) != file_sha256(copied):
                    raise ValueError(f'Mevcut görüntü kopyası kaynakla farklı: {copied}')
            else:
                shutil.copy2(original, copied)
            manifest.append({'partition': role, 'sample_index': int(i), 'source_path': paths[i], 'export_path': copied.relative_to(directory).as_posix(), 'class': classes[int(labels[i])], 'group': str(groups[i]) if groups is not None else ''})
            if number % 500 == 0:
                print(f'{role}: {number}/{len(indices)} kopya kontrol edildi', flush=True)
        counts.append({'partition': role, 'total': len(indices), 'percent': 100 * len(indices) / len(labels), **{name: int(np.sum(labels[indices] == i)) for i, name in enumerate(classes)}})
    write_csv(directory / 'sample_index.csv', [{'sample_index': i, 'path': path, 'class': classes[int(labels[i])], 'image_sha256': audit['image_sha256'][i]} for i, path in enumerate(paths)])
    write_csv(directory / 'split_manifest.csv', manifest)
    write_csv(directory / 'class_counts.csv', counts)
    (directory / 'data_audit.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')
    print('Görüntü kopyaları ve indeksler hazır:', directory, flush=True)
    return (split, counts, metadata)

def export_test_predictions(probabilities, labels, names, classes, test_indices, test_paths, output, positive_name, tie_rule):
    """Stream image-level predictions for all selected comparisons to CSV."""
    validate(probabilities, labels, names, classes)
    test_indices = np.asarray(test_indices)
    if test_indices.ndim != 1 or not np.issubdtype(test_indices.dtype, np.integer):
        raise ValueError('test_indices tek boyutlu tam sayı dizisi olmalı.')
    if len(test_indices) != len(labels) or len(test_paths) != len(labels):
        raise ValueError('Test indeksleri, dosya yolları ve tahmin sayıları uyuşmuyor.')
    if len(np.unique(test_indices)) != len(labels) or len(set(test_paths)) != len(labels):
        raise ValueError('Test indekslerinde veya görüntü yollarında tekrar var.')
    if positive_name not in classes:
        raise ValueError('Pozitif sınıf sınıf listesinde bulunamadı.')
    positive = classes.index(positive_name)
    directory = output / 'test_prediction_reports'
    directory.mkdir(parents=True, exist_ok=True)
    fields = ['test_position', 'sample_index', 'image_name', 'image_path', 'split_image_path', 'models', 'voting', 'true_class', 'predicted_class', 'is_correct', 'error_type', 'positive_class', 'positive_score', 'predicted_class_score', 'score_type', 'hard_vote_tie', 'tie_rule'] + ['score_' + name for name in classes]

    def open_report(filename):
        handle = (directory / filename).open('w', newline='', encoding='utf-8-sig')
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        return (handle, writer)
    all_handle, all_writer = open_report('test_predictions_all.csv')
    error_handle, error_writer = open_report('misclassified_all.csv')
    total_rows = error_rows = 0
    try:
        for indices in experiment_subsets(names):
            size = len(indices)
            selected_names = [names[i] for i in indices]
            selected = probabilities[list(indices)]
            for strategy in ('soft', 'hard'):
                predicted, scores, _ = vote(selected, strategy, tie_rule)
                ties = (selected.argmax(axis=2) == 0).sum(axis=0) == size / 2
                detail_handles = []
                if size == len(names):
                    prefix = 'full_ensemble_' + strategy
                elif size == 1 and strategy == 'soft':
                    prefix = 'single_' + selected_names[0]
                else:
                    prefix = None
                if prefix:
                    h, w = open_report(prefix + '_predictions.csv')
                    eh, ew = open_report(prefix + '_misclassified.csv')
                    detail_handles = [h, eh]
                try:
                    for position, (truth, guess) in enumerate(zip(labels, predicted)):
                        correct = int(truth) == int(guess)
                        error = '' if correct else 'FN' if int(truth) == positive else 'FP'
                        path = str(test_paths[position])
                        row = {'test_position': position, 'sample_index': int(test_indices[position]), 'image_name': Path(path).name, 'image_path': path, 'split_image_path': 'test/' + path, 'models': '+'.join(selected_names), 'voting': strategy, 'true_class': classes[int(truth)], 'predicted_class': classes[int(guess)], 'is_correct': correct, 'error_type': error, 'positive_class': positive_name, 'positive_score': float(scores[position, positive]), 'predicted_class_score': float(scores[position, int(guess)]), 'score_type': 'mean_probability' if strategy == 'soft' else 'vote_fraction', 'hard_vote_tie': bool(ties[position]) if strategy == 'hard' else False, 'tie_rule': tie_rule if strategy == 'hard' else 'lowest_argmax'}
                        row.update({'score_' + name: float(scores[position, i]) for i, name in enumerate(classes)})
                        all_writer.writerow(row)
                        total_rows += 1
                        if not correct:
                            error_writer.writerow(row)
                            error_rows += 1
                        if prefix:
                            w.writerow(row)
                            if not correct:
                                ew.writerow(row)
                finally:
                    for handle in detail_handles:
                        handle.close()
    finally:
        all_handle.close()
        error_handle.close()
    (directory / 'README.txt').write_text("image_path: BASE_DIR köküne göre kaynak görüntü yolu.\nsplit_image_path: SPLIT_DIR köküne göre test kopyası yolu.\nsample_index: ortak veri dizisindeki indeks; test_position: predictions.npz test sırası.\nis_correct=False: hatalı sınıflandırma. FN/FP, positive_class'a göre tanımlıdır.\nSoft skorlar ortalama olasılık; hard skorlar oy oranıdır ve güven olasılığı değildir.\nHard tie soft kuralıyla çözülürse seçilen sınıfın oy oranı 0.5 olabilir.\nmisclassified_all.csv içinde aynı görüntü birden fazla deneyde yer alabilir.\n", encoding='utf-8')
    print(f'Görüntü bazlı {total_rows} tahmin satırı ve {error_rows} hatalı satır kaydedildi: {directory}')
    return directory

## 6. Create/Load Shared Indices and Copy Images into Split Directories

On the first run, the 80/10/10 split indices are generated and saved. On subsequent runs, if SPLIT_DIR remains unchanged, the indices are loaded from disk. The source images are checked for consistency. The total, training, validation, and test sample counts are displayed for each class.

For a new training run, select a different OUTPUT_DIR. Keep SPLIT_DIR unchanged. To generate a new split, select a new SPLIT_DIR. Since the indices are stored together with the data list, do not modify the source data.

In [5]:
if not args.evaluate_only:
    if len(set(args.classes)) != 2 or args.positive_class not in args.classes:
        raise ValueError("İki farklı sınıf ve geçerli pozitif sınıf gerekli.")
    if (args.train_fraction, args.val_fraction, args.test_fraction) != (0.80, 0.10, 0.10):
        raise ValueError("Bu notebook sabit 80/10/10 bölme kullanır.")
    if min(args.epochs, args.batch_size, args.predict_batch_size, args.deit_feature_batch_size) < 1:
        raise ValueError("Epoch ve batch değerleri pozitif olmalı.")
    output.mkdir(parents=True, exist_ok=True)
    if (output / "predictions.npz").exists() or list(output.glob("*.h5")):
        raise ValueError("Yeni eğitim için başka OUTPUT_DIR seçin; aynı SPLIT_DIR'yi koruyun. Rapor için EVALUATE_ONLY=True.")
    images, labels, paths, groups, classes, audit = load_dataset(args)
    split, counts, split_metadata = prepare_reusable_split(args, labels, paths, groups, classes, audit)
    train_idx, val_idx, test_idx = split
    config = {"protocol": "single_split_80_10_10", "base_dir": str(args.base_dir.resolve()),
              "split_dir": str(args.split_dir.resolve()), "classes": classes, "seed": args.seed,
              "epochs": args.epochs, "batch_size": args.batch_size,
              "model_names": MODEL_NAMES, "transformer_checkpoints": TRANSFORMER_SPECS,
              "predict_batch_size": args.predict_batch_size,
              "deit_feature_batch_size": args.deit_feature_batch_size,
              "split_seed": split_metadata["split_seed"],
              "dataset_fingerprint": split_metadata["dataset_fingerprint"],
              "indices_sha256": split_metadata["indices_sha256"]}
    study_id = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
    study_path = output / "study.json"
    if study_path.exists() and json.loads(study_path.read_text()).get("study_id") != study_id:
        raise ValueError("Bu OUTPUT_DIR başka eğitim ayarlarına ait; yeni OUTPUT_DIR seçin.")
    config["study_id"] = study_id
    study_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
    # Eğitim sonuçlarının yanında da split kayıtlarının bir kopyası tutulur.
    for filename in ["split_indices.npz", "split_metadata.json", "sample_index.csv",
                     "split_manifest.csv", "class_counts.csv", "data_audit.json"]:
        source_file = args.split_dir.resolve() / filename
        target_file = output / filename
        if source_file != target_file:
            shutil.copy2(source_file, target_file)
    import pandas as pd
    display(pd.DataFrame(counts))
    print("Tüm modeller aynı kayıtlı split'i kullanacak.")
else:
    print("Değerlendirme modu: veri yükleme ve kopyalama atlandı.")


Yeni ortak indeksler oluşturuldu ve kaydedildi: /home/tubitak1001/Desktop/arzu/data_arzu_14models_split_80_10_10/split_indices.npz
train: 500/3143 kopya kontrol edildi
train: 1000/3143 kopya kontrol edildi
train: 1500/3143 kopya kontrol edildi
train: 2000/3143 kopya kontrol edildi
train: 2500/3143 kopya kontrol edildi
train: 3000/3143 kopya kontrol edildi
Görüntü kopyaları ve indeksler hazır: /home/tubitak1001/Desktop/arzu/data_arzu_14models_split_80_10_10


,partition,total,percent,damaged,undamaged
0,all,3929,100.000000,2678,1251
1,train,3143,79.994910,2142,1001
2,val,393,10.002545,268,125
3,test,393,10.002545,268,125


Tüm modeller aynı kayıtlı split'i kullanacak.


## 7. TensorFlow ve PyTorch eğitim ortamını hazırla

In [6]:
if not args.evaluate_only:
    init_training()
    import torch
    from transformers import AutoImageProcessor, AutoModel

    if not tf.config.list_physical_devices("GPU"):
        raise RuntimeError("TensorFlow GPU bulamadı. DEIT GPU — TensorFlow + Torch kernel'ini seçin.")
    if args.transformer_device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("Torch GPU bulamadı; önce GPU ortamını kontrol edin.")

    tf.config.set_soft_device_placement(False)
    with tf.device("/GPU:0"):
        tf_test_layer = tf.keras.layers.Conv2D(16, 3, padding="same")
        tf_test_input = tf.random.normal((1, 224, 224, 3))
        tf_test_output = tf_test_layer(tf_test_input)
        _ = tf_test_output.numpy()
    print("TensorFlow GPU testi başarılı:", tf_test_output.device)
    del tf_test_layer, tf_test_input, tf_test_output
    tf.keras.backend.clear_session()

    if args.transformer_device == "cuda":
        with torch.inference_mode():
            torch_test_layer = torch.nn.Conv2d(3, 16, 3, padding=1).cuda()
            torch_test_input = torch.randn(1, 3, 224, 224, device="cuda")
            torch_test_output = torch_test_layer(torch_test_input)
            torch.cuda.synchronize()
        print("Torch GPU testi başarılı:", torch.cuda.get_device_name(0))
        del torch_test_layer, torch_test_input, torch_test_output
        torch.cuda.empty_cache()
    print("Eğitim ortamı hazır.")
else:
    print("Değerlendirme modu: model yükleme atlandı.")


2026-10-03 18:13:19.132970: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-03 18:13:19.162380: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-03 18:13:19.526562: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2026-10-03 18:13:19.778541: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/t

TensorFlow GPU testi başarılı: /job:localhost/replica:0/task:0/device:GPU:0
Torch GPU testi başarılı: NVIDIA GeForce RTX 4090
Eğitim ortamı hazır.


## ## 8. Train the Fourteen Models Sequentially

Each model is trained once. The CNN models and Transformer classification heads use the same train/validation/test indices. Each model’s test predictions are saved separately after training.

if not args.evaluate_only:
    if (output / "predictions.npz").exists() or list(output.glob("*.h5")):
        raise ValueError("Eğitim dosyaları zaten mevcut; yeni bir OUTPUT_DIR ile başlayın.")
    probabilities = train_single_split(images, labels, split, classes, args, output)
    provenance = {"protocol": "single_split_80_10_10", "study_id": study_id,
                  "seed": args.seed, "model_names": MODEL_NAMES, "ab_mode": args.ab_mode,
                  "transformer_checkpoints": TRANSFORMER_SPECS,
                  "transformer_backend": "pytorch frozen backbone; keras trained head", "weights": "EarlyStopping val_loss; original training policy"}
    np.savez_compressed(output / "predictions.npz", probabilities=probabilities,
                        labels=labels[test_idx], model_names=np.asarray(MODEL_NAMES),
                        class_names=np.asarray(classes), train_indices=train_idx,
                        validation_indices=val_idx, test_indices=test_idx,
                        test_paths=np.asarray([paths[i] for i in test_idx]),
                        provenance=json.dumps(provenance))
    print("Tek split model tahminleri kaydedildi.")


In [7]:
import gc
import torch

# Kalan Transformer özelliklerini RAM/CPU üzerinde çıkar.
args.transformer_device = "cpu"
TRANSFORMER_DEVICE = "cpu"

gc.collect()
tf.keras.backend.clear_session()
torch.cuda.empty_cache()

all_model_names = list(MODEL_NAMES)
cached = {}

# Tamamlanan modellerin kaydedilmiş test tahminlerini yükle.
for name in all_model_names:
    file = output / f"{name}_test_predictions.npz"

    if file.exists():
        with np.load(file, allow_pickle=False) as data:
            if not np.array_equal(data["test_indices"], test_idx):
                raise ValueError(f"{name}: test indeksleri uyuşmuyor.")
            if not np.array_equal(data["labels"], labels[test_idx]):
                raise ValueError(f"{name}: test etiketleri uyuşmuyor.")

            cached[name] = data["probabilities"].copy()

        print("Tamamlandı, tekrar eğitilmeyecek:", name)

remaining = [name for name in all_model_names if name not in cached]
print("Çalıştırılacak modeller:", remaining)

# Yalnızca tahmini kaydedilmemiş modelleri çalıştır.
if remaining:
    MODEL_NAMES = remaining
    try:
        train_single_split(images, labels, split, classes, args, output)
    finally:
        MODEL_NAMES = all_model_names

# Bütün modellerin tahminlerini orijinal sırayla birleştir.
predictions = []
for name in all_model_names:
    with np.load(
        output / f"{name}_test_predictions.npz",
        allow_pickle=False
    ) as data:
        predictions.append(data["probabilities"].copy())

probabilities = np.stack(predictions)

provenance = {
    "protocol": "single_split_80_10_10",
    "study_id": study_id,
    "seed": args.seed,
    "model_names": all_model_names,
    "ab_mode": args.ab_mode,
    "transformer_checkpoints": TRANSFORMER_SPECS,
    "transformer_backend": "pytorch frozen backbone; keras trained head",
    "resumed_feature_device": "cpu",
    "resumed_models": remaining,
    "weights": "EarlyStopping val_loss; original training policy",
}

np.savez_compressed(
    output / "predictions.npz",
    probabilities=probabilities,
    labels=labels[test_idx],
    model_names=np.asarray(all_model_names),
    class_names=np.asarray(classes),
    train_indices=train_idx,
    validation_indices=val_idx,
    test_indices=test_idx,
    test_paths=np.asarray([paths[i] for i in test_idx]),
    provenance=json.dumps(provenance),
)

print("14 modelin test tahminleri hazır. Rapor hücrelerine geçebilirsiniz.")

Çalıştırılacak modeller: ['VGG19', 'DenseNet201', 'DenseNet169', 'ConvNeXtBase', 'VGG16', 'ResNet50', 'NASNetMobile', 'Xception', 'DeiT', 'ViTBase', 'ViTLarge', 'ViTHuge', 'SwinT', 'CvT']

Model 1/14: VGG19
Epoch 1/30


I0000 00:00:1791040415.818596  650285 service.cc:145] XLA service 0x7f577834e000 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1791040415.818613  650285 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
2026-10-03 18:13:35.821405: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1791040415.863555  650285 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


99/99 [==============================] - ETA: 0s - loss: 0.4083 - accuracy: 0.8078
Epoch 1: val_accuracy improved from -inf to 0.70992, saving model to VGG19.h5
99/99 [==============================] - 20s 98ms/step - loss: 0.4083 - accuracy: 0.8078 - val_loss: 2.8580 - val_accuracy: 0.7099 - lr: 1.0000e-04
Epoch 2/30


/home/tubitak1001/projects/deit_env/lib/python3.11/site-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


98/99 [============================>.] - ETA: 0s - loss: 0.2557 - accuracy: 0.8983
Epoch 2: val_accuracy did not improve from 0.70992
99/99 [==============================] - 8s 83ms/step - loss: 0.2558 - accuracy: 0.8982 - val_loss: 4.7663 - val_accuracy: 0.6997 - lr: 1.0000e-04
Epoch 3/30
98/99 [============================>.] - ETA: 0s - loss: 0.2429 - accuracy: 0.9066
Epoch 3: val_accuracy did not improve from 0.70992
99/99 [==============================] - 8s 82ms/step - loss: 0.2430 - accuracy: 0.9065 - val_loss: 5.1251 - val_accuracy: 0.6819 - lr: 1.0000e-04
Epoch 4/30
98/99 [============================>.] - ETA: 0s - loss: 0.2318 - accuracy: 0.9082
Epoch 4: val_accuracy did not improve from 0.70992
99/99 [==============================] - 8s 82ms/step - loss: 0.2314 - accuracy: 0.9084 - val_loss: 2.5642 - val_accuracy: 0.6819 - lr: 1.0000e-04
Epoch 5/30
98/99 [============================>.] - ETA: 0s - loss: 0.1956 - accuracy: 0.9267
Epoch 5: val_accuracy improved from 0.709

...

ViTBase: 16/3929 özellik çıkarıldı
ViTBase: 336/3929 özellik çıkarıldı
ViTBase: 656/3929 özellik çıkarıldı
ViTBase: 976/3929 özellik çıkarıldı
ViTBase: 1296/3929 özellik çıkarıldı
ViTBase: 1616/3929 özellik çıkarıldı
ViTBase: 1936/3929 özellik çıkarıldı
ViTBase: 2256/3929 özellik çıkarıldı
ViTBase: 2576/3929 özellik çıkarıldı
ViTBase: 2896/3929 özellik çıkarıldı
ViTBase: 3216/3929 özellik çıkarıldı
ViTBase: 3536/3929 özellik çıkarıldı
ViTBase: 3856/3929 özellik çıkarıldı
ViTBase: 3929/3929 özellik çıkarıldı
Epoch 1/30
91/99 [==========================>...] - ETA: 0s - loss: 0.4704 - accuracy: 0.7830
Epoch 1: val_accuracy improved from -inf to 0.91603, saving model to ViTBase_deit_clf.h5
99/99 [==============================] - 1s 5ms/step - loss: 0.4559 - accuracy: 0.7913 - val_loss: 0.3881 - val_accuracy: 0.9160 - lr: 1.0000e-04
Epoch 2/30
64/99 [==================>...........] - ETA: 0s - loss: 0.2785 - accuracy: 0.8784
Epoch 2: val_accuracy improved from 0.91603 to 0.94911, saving m

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


ViTLarge: 2/3929 özellik çıkarıldı
ViTLarge: 42/3929 özellik çıkarıldı
ViTLarge: 82/3929 özellik çıkarıldı
ViTLarge: 122/3929 özellik çıkarıldı
ViTLarge: 162/3929 özellik çıkarıldı
ViTLarge: 202/3929 özellik çıkarıldı
ViTLarge: 242/3929 özellik çıkarıldı
ViTLarge: 282/3929 özellik çıkarıldı
ViTLarge: 322/3929 özellik çıkarıldı
ViTLarge: 362/3929 özellik çıkarıldı
ViTLarge: 402/3929 özellik çıkarıldı
ViTLarge: 442/3929 özellik çıkarıldı
ViTLarge: 482/3929 özellik çıkarıldı
ViTLarge: 522/3929 özellik çıkarıldı
ViTLarge: 562/3929 özellik çıkarıldı
ViTLarge: 602/3929 özellik çıkarıldı
ViTLarge: 642/3929 özellik çıkarıldı
ViTLarge: 682/3929 özellik çıkarıldı
ViTLarge: 722/3929 özellik çıkarıldı
ViTLarge: 762/3929 özellik çıkarıldı
ViTLarge: 802/3929 özellik çıkarıldı
ViTLarge: 842/3929 özellik çıkarıldı
ViTLarge: 882/3929 özellik çıkarıldı
ViTLarge: 922/3929 özellik çıkarıldı
ViTLarge: 962/3929 özellik çıkarıldı
ViTLarge: 1002/3929 özellik çıkarıldı
ViTLarge: 1042/3929 özellik çıkarıldı
ViT

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


ViTHuge: 1/3929 özellik çıkarıldı
ViTHuge: 21/3929 özellik çıkarıldı
ViTHuge: 41/3929 özellik çıkarıldı
ViTHuge: 61/3929 özellik çıkarıldı
ViTHuge: 81/3929 özellik çıkarıldı
ViTHuge: 101/3929 özellik çıkarıldı
ViTHuge: 121/3929 özellik çıkarıldı
ViTHuge: 141/3929 özellik çıkarıldı
ViTHuge: 161/3929 özellik çıkarıldı
ViTHuge: 181/3929 özellik çıkarıldı
ViTHuge: 201/3929 özellik çıkarıldı
ViTHuge: 221/3929 özellik çıkarıldı
ViTHuge: 241/3929 özellik çıkarıldı
ViTHuge: 261/3929 özellik çıkarıldı
ViTHuge: 281/3929 özellik çıkarıldı
ViTHuge: 301/3929 özellik çıkarıldı
ViTHuge: 321/3929 özellik çıkarıldı
ViTHuge: 341/3929 özellik çıkarıldı
ViTHuge: 361/3929 özellik çıkarıldı
ViTHuge: 381/3929 özellik çıkarıldı
ViTHuge: 401/3929 özellik çıkarıldı
ViTHuge: 421/3929 özellik çıkarıldı
ViTHuge: 441/3929 özellik çıkarıldı
ViTHuge: 461/3929 özellik çıkarıldı
ViTHuge: 481/3929 özellik çıkarıldı
ViTHuge: 501/3929 özellik çıkarıldı
ViTHuge: 521/3929 özellik çıkarıldı
ViTHuge: 541/3929 özellik çıkarıld

## 9. Calculating and reporting ablation results

In [8]:
rows = evaluate_single_split(args, output)


Component ablation (key comparisons):
VGG19                                                   soft Acc=0.9491 Sens=0.9478 Spec=0.9520 F1=0.9621 MCC=0.8856 deltaAcc=-0.0407
VGG19                                                   hard Acc=0.9491 Sens=0.9478 Spec=0.9520 F1=0.9621 MCC=0.8856 deltaAcc=-0.0382
DenseNet201                                             soft Acc=0.9695 Sens=0.9701 Spec=0.9680 F1=0.9774 MCC=0.9305 deltaAcc=-0.0204
DenseNet201                                             hard Acc=0.9695 Sens=0.9701 Spec=0.9680 F1=0.9774 MCC=0.9305 deltaAcc=-0.0178
DenseNet169                                             soft Acc=0.9746 Sens=0.9739 Spec=0.9760 F1=0.9812 MCC=0.9421 deltaAcc=-0.0153
DenseNet169                                             hard Acc=0.9746 Sens=0.9739 Spec=0.9760 F1=0.9812 MCC=0.9421 deltaAcc=-0.0127
ConvNeXtBase                                            soft Acc=0.9822 Sens=0.9813 Spec=0.9840 F1=0.9869 MCC=0.9593 deltaAcc=-0.0076
ConvNeXtBase           

In [11]:
from pathlib import Path
import itertools
import pandas as pd

AB_MODE = "all"
args.ab_mode = "all"


# Bütün boş olmayan model kombinasyonları
def experiment_subsets(names):
    return [
        combination
        for size in range(1, len(names) + 1)
        for combination in itertools.combinations(
            range(len(names)), size
        )
    ]


result_root = Path(output).resolve()

# Önce 5-fold sonuçları, yoksa tek-split sonucu
fold_dirs = sorted(
    directory
    for directory in result_root.glob("fold_*")
    if directory.is_dir()
)

if fold_dirs:
    targets = [
        (directory.name, directory)
        for directory in fold_dirs
    ]
else:
    targets = [("single_split", result_root)]

# Eksik tahmin dosyası varsa hesaplamaya başlamadan bildir.
missing = [
    str(directory / "predictions.npz")
    for _, directory in targets
    if not (directory / "predictions.npz").is_file()
]

if missing:
    raise FileNotFoundError(
        "Önce model eğitimleri tamamlanmalı. Eksik dosyalar:\n"
        + "\n".join(missing)
    )

tables = []

for fold_name, directory in targets:
    print(f"\n{fold_name}: bütün kombinasyonlar hesaplanıyor...")

    # Notebook'taki fonksiyon:
    # soft/hard metrikleri ve görüntü bazındaki raporları kaydeder.
    rows = evaluate_single_split(args, directory)

    table = pd.DataFrame(rows)
    table.insert(0, "fold", fold_name)
    tables.append(table)

    print(f"{fold_name}: {len(table):,} sonuç satırı hazır.")

all_results = pd.concat(tables, ignore_index=True)

all_results.to_csv(
    result_root / "ablation_all_combinations.csv",
    index=False,
    encoding="utf-8-sig",
)

# Her kombinasyon ve voting türü için fold ortalaması/std
if len(targets) > 1:
    metrics = [
        "accuracy", "balanced_accuracy",
        "sensitivity", "specificity", "precision",
        "f1_positive", "f1_macro", "mcc", "auc",
    ]

    summary = (
        all_results
        .groupby(["models", "n_models", "voting"])[metrics]
        .agg(["mean", "std"])
    )

    summary.columns = [
        f"{metric}_{stat}"
        for metric, stat in summary.columns
    ]

    summary.reset_index().to_csv(
        result_root / "ablation_all_combinations_mean_std.csv",
        index=False,
        encoding="utf-8-sig",
    )

print("\nToplam sonuç satırı:", len(all_results))
print("Sonuç klasörü:", result_root)


single_split: bütün kombinasyonlar hesaplanıyor...

Component ablation (key comparisons):
VGG19                                                   soft Acc=0.9491 Sens=0.9478 Spec=0.9520 F1=0.9621 MCC=0.8856 deltaAcc=-0.0407
VGG19                                                   hard Acc=0.9491 Sens=0.9478 Spec=0.9520 F1=0.9621 MCC=0.8856 deltaAcc=-0.0382
DenseNet201                                             soft Acc=0.9695 Sens=0.9701 Spec=0.9680 F1=0.9774 MCC=0.9305 deltaAcc=-0.0204
DenseNet201                                             hard Acc=0.9695 Sens=0.9701 Spec=0.9680 F1=0.9774 MCC=0.9305 deltaAcc=-0.0178
DenseNet169                                             soft Acc=0.9746 Sens=0.9739 Spec=0.9760 F1=0.9812 MCC=0.9421 deltaAcc=-0.0153
DenseNet169                                             hard Acc=0.9746 Sens=0.9739 Spec=0.9760 F1=0.9812 MCC=0.9421 deltaAcc=-0.0127
ConvNeXtBase                                            soft Acc=0.9822 Sens=0.9813 Spec=0.9840 F1=0.9869

## 10. Saving results

`ablation_results.csv`: selected single-split comparisons. Each `delta_*_vs_full` value is calculated as the subset metric minus the full-ensemble metric for the same voting strategy.

`split_indices.npz`: shared train/validation/test indices. `sample_index.csv`: mapping between indices and images. `class_counts.csv`: class counts for the full dataset and each split.

Model files and `predictions.npz` are saved directly in the output directory. Hard-voting AUC is calculated from the proportion of positive votes, while soft-voting AUC is calculated from the mean positive-class probability.

`SPLIT_DIR/train`, `SPLIT_DIR/val`, and `SPLIT_DIR/test` contain copies of the images. The same data split can be reused for new training runs by changing `OUTPUT_DIR` while keeping `SPLIT_DIR` unchanged.

In [9]:
import pandas as pd
results_df = pd.read_csv(output / "ablation_results.csv")
key_df = results_df.loc[results_df["experiment"] != "subset"].copy()
display(key_df[["experiment", "models", "voting", "accuracy", "sensitivity",
                "specificity", "precision", "f1_positive", "f1_macro", "mcc", "auc",
                "delta_accuracy_vs_full"]].round(4))
print(f"Tek split ablasyon sonuçları: {len(results_df)}")


,experiment,models,voting,accuracy,sensitivity,specificity,precision,f1_positive,f1_macro,mcc,auc,delta_accuracy_vs_full
0,single_model,VGG19,soft,0.9491,0.9478,0.952,0.9769,0.9621,0.9423,0.8856,0.9899,-0.0407
1,single_model,VGG19,hard,0.9491,0.9478,0.952,0.9769,0.9621,0.9423,0.8856,0.9499,-0.0382
2,single_model,DenseNet201,soft,0.9695,0.9701,0.968,0.9848,0.9774,0.9651,0.9305,0.9953,-0.0204
3,single_model,DenseNet201,hard,0.9695,0.9701,0.968,0.9848,0.9774,0.9651,0.9305,0.9691,-0.0178
4,single_model,DenseNet169,soft,0.9746,0.9739,0.976,0.9886,0.9812,0.9709,0.9421,0.9978,-0.0153
...,...,...,...,...,...,...,...,...,...,...,...,...
61,Transformer_only,DeiT+ViTBase+ViTLarge+ViTHuge+SwinT+CvT,hard,0.9898,0.9925,0.984,0.9925,0.9925,0.9883,0.9765,0.9996,0.0025
62,original_CTVE,VGG19+DenseNet201+DenseNet169+ConvNeXtBase+DeiT,soft,0.9847,0.9813,0.992,0.9962,0.9887,0.9825,0.9654,0.9993,-0.0051
63,original_CTVE,VGG19+DenseNet201+DenseNet169+ConvNeXtBase+DeiT,hard,0.9847,0.9813,0.992,0.9962,0.9887,0.9825,0.9654,0.9975,-0.0025
64,original_CVE,VGG19+DenseNet201+DenseNet169+ConvNeXtBase,soft,0.9873,0.9851,0.992,0.9962,0.9906,0.9854,0.9710,0.9989,-0.0025


Tek split ablasyon sonuçları: 66


## 11. Misclassified Test Images

Reports are saved in `OUTPUT_DIR/test_prediction_reports/`:

- `test_predictions_all.csv`: predictions for each test image across the selected comparisons.
- `misclassified_all.csv`: misclassified predictions across all comparisons.
- `full_ensemble_soft_misclassified.csv` and `full_ensemble_hard_misclassified.csv`: misclassifications by the full ensemble.
- `single_<model>_misclassified.csv`: misclassifications by individual models.

The columns include the image name/path, test order, shared sample index, true class, predicted class, correctness indicator, FN/FP indicators, and class scores. Soft-voting scores are mean class probabilities; hard-voting scores are vote proportions.

If training has already been completed, set **`EVALUATE_ONLY=True` using the same `OUTPUT_DIR`**. Run the configuration and function-definition cells, followed by evaluation cell 9. Retraining is not required. The cell below displays the full ensemble’s soft-voting misclassifications.## 

In [10]:
import pandas as pd
report_dir = output / "test_prediction_reports"
errors_df = pd.read_csv(report_dir / "full_ensemble_soft_misclassified.csv")
print(f"Tam topluluk / soft voting: {len(errors_df)} hatalı test görüntüsü")
display(errors_df[["sample_index", "image_name", "image_path", "true_class",
                   "predicted_class", "error_type", "positive_score"]].head(50))
# Hard voting için: full_ensemble_hard_misclassified.csv
# Tek VGG19 için: single_VGG19_misclassified.csv


Tam topluluk / soft voting: 4 hatalı test görüntüsü


,sample_index,image_name,image_path,true_class,predicted_class,error_type,positive_score
0,1275,2270.jpg,damaged/2270.jpg,damaged,undamaged,FN,0.464450
1,1959,312.jpg,damaged/312.jpg,damaged,undamaged,FN,0.219761
2,3769,852.jpg,undamaged/852.jpg,undamaged,damaged,FP,0.583963
3,1482,2495.jpg,damaged/2495.jpg,damaged,undamaged,FN,0.421951
